In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import os


from model.loss_function.siamese.custom_siamese_loss import CustomSiameseLoss
from model.architecture.siamese_model import SiameseAHPModel
from data_augmentation.matrices_loader import MatricesLoader
from data_augmentation.datasets.siamese_AHP_matrix_dataset import SiameseAHPMatrixDataset
from data_augmentation.generator.target_cr_generator import TargetCRMatricesGenerator
from utils.phase import Phase

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device used: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device used: cpu


In [ ]:
IS_UNIFORM = True
CRITERIA_NUM = 5
CONSISTENCY_RATES = [0.01, 0.02, 0.03, 0.04, 0.05, 0.07, 0.08, 0.1, 0.12, 0.14, 0.15, 
                     0.17, 0.18, 0.2, 0.23, 0.27, 0.3, 0.33, 0.37, 0.4]

In [ ]:
def load_train_dataset(loader: MatricesLoader):
    all_matrices = []
    all_comparison_matrices = []
    all_weights = []

    for cr in CONSISTENCY_RATES:
        matrices, weights = loader.load_noisy_cr_matrices(Phase.TRAIN, cr, IS_UNIFORM)
        generator = TargetCRMatricesGenerator(matrices.shape[0], matrices.shape[1], cr)
        comparison_matrices = generator.from_weights(weights)

        all_matrices.append(matrices)
        all_comparison_matrices.append(comparison_matrices)
        all_weights.append(weights)
    
    final_matrices = np.concatenate(all_matrices, axis=0)
    final_weights = np.concatenate(all_weights, axis=0)
    final_comparison_matrices = np.concatenate(all_comparison_matrices, axis=0)
    dataset = SiameseAHPMatrixDataset(final_matrices, final_comparison_matrices, final_weights)

    print(f"Train dataset was successfully created and {len(dataset)} matrices was uploaded")
    return dataset


def load_valid_dataset(loader: MatricesLoader):
    all_matrices = []
    all_comparison_matrices = []
    all_weights = []

    for cr in CONSISTENCY_RATES:
        matrices, weights = loader.load_noisy_cr_matrices(Phase.VALIDATION, cr, IS_UNIFORM)
        generator = TargetCRMatricesGenerator(matrices.shape[0], matrices.shape[1], cr)
        comparison_matrices = generator.from_weights(weights)

        all_matrices.append(matrices)
        all_comparison_matrices.append(comparison_matrices)
        all_weights.append(weights)
    
    final_matrices = np.concatenate(all_matrices, axis=0)
    final_weights = np.concatenate(all_weights, axis=0)
    final_comparison_matrices = np.concatenate(all_comparison_matrices, axis=0)
    dataset = SiameseAHPMatrixDataset(final_matrices, final_comparison_matrices, final_weights)

    print(f"Validation dataset was successfully created and {len(dataset)} matrices was uploaded")
    return dataset


def save_model_weights(model: torch.nn.Module, model_type_name: str, file_name: str):
    base_dir = "models"
    dir_path = os.path.join(base_dir, model_type_name)

    if not os.path.exists(dir_path):
        os.makedirs(dir_path)

    file_path = os.path.join(dir_path, file_name)
    torch.save(model.state_dict(), file_path)
    print(f"Model: {file_name.split('.')[0]} was saved in {dir_path}")


In [ ]:
loader = MatricesLoader()
dataset = load_train_dataset(loader)

Successfuly uploaded 1000 noised matrices (level: c001).
Successfuly uploaded 1000 noised matrices (level: c002).
Successfuly uploaded 1000 noised matrices (level: c003).
Successfuly uploaded 1000 noised matrices (level: c004).
Successfuly uploaded 1000 noised matrices (level: c005).
Successfuly uploaded 1000 noised matrices (level: c007).
Successfuly uploaded 1000 noised matrices (level: c008).
Successfuly uploaded 1000 noised matrices (level: c010).
Successfuly uploaded 1000 noised matrices (level: c012).
Successfuly uploaded 1000 noised matrices (level: c014).
Successfuly uploaded 1000 noised matrices (level: c015).
Successfuly uploaded 1000 noised matrices (level: c017).
Successfuly uploaded 1000 noised matrices (level: c018).
Successfuly uploaded 1000 noised matrices (level: c020).
Successfuly uploaded 1000 noised matrices (level: c023).
Successfuly uploaded 1000 noised matrices (level: c027).
Successfuly uploaded 1000 noised matrices (level: c030).
Successfuly uploaded 1000 noise

In [6]:
def train_model(model, dataset, epochs=100, batch_size=32, lr=0.001):
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    criterion = CustomSiameseLoss(lambda_cop=1.0, lambda_rec=0.5, lambda_stab=0.2).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = []
    model.train()

    for epoch in range(epochs):
        epoch_loss = 0.0
        
        for batch_idx, (m1_batch, m2_batch, weights_batch) in enumerate(dataloader):
            m1_batch = m1_batch.to(device)
            m2_batch = m2_batch.to(device)
            weights_batch = weights_batch.to(device)

            logits1, logits2 = model(m1_batch, m2_batch)
            loss = criterion(logits1, logits2, weights_batch, m1_batch.squeeze(1), m2_batch.squeeze(1))

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(dataloader)
        history.append(avg_loss)

        if (epoch + 1) % 10 == 0:
            print(f"Epoka [{epoch+1}/{epochs}] | Średnia Strata: {avg_loss:.6f}")

    return model, history

In [8]:
model = SiameseAHPModel(n_criteria=CRITERIA_NUM)
trained_model, loss_history = train_model(model, dataset)

save_model_weights(model, "siamese", "basic_model.pth")

Epoka [10/100] | Średnia Strata: 0.019111
Epoka [20/100] | Średnia Strata: 0.018548
Epoka [30/100] | Średnia Strata: 0.018174
Epoka [40/100] | Średnia Strata: 0.017955
Epoka [50/100] | Średnia Strata: 0.017859
Epoka [60/100] | Średnia Strata: 0.017796
Epoka [70/100] | Średnia Strata: 0.017704
Epoka [80/100] | Średnia Strata: 0.017720
Epoka [90/100] | Średnia Strata: 0.017636
Epoka [100/100] | Średnia Strata: 0.017633
Model: basic_model was saved in models/siamese
